# Feature Selection

## Learning Objectives
- Understand why $n \gg m$ causes overfitting even with simple models (VC dimension = $O(n)$)
- Implement **forward search** and **backward search** (wrapper methods)
- Implement **filter feature selection** using mutual information and correlation scores
- Understand the KL-divergence interpretation of mutual information
- Compare wrapper vs. filter methods in terms of accuracy and computational cost

## Problem Statement

Given $n$ features where $n \gg m$, find a small subset $\mathcal{F} \subseteq \{1, \ldots, n\}$ that preserves predictive accuracy.

**Why feature selection matters:**
A linear classifier over $n$ features has $\text{VC}(\mathcal{H}) = O(n)$, so sample complexity is $O(n)$.
If $n \gg m$, the model will overfit even with a simple hypothesis class.

**Two families of methods:**

| Family | Idea | Cost |
|---|---|---|
| **Wrapper** | Use the learning algorithm to evaluate each feature subset | $O(n^2)$ calls to learner |
| **Filter** | Score each feature independently; pick top $k$ | $O(n)$ — much cheaper |

**Mutual information (filter score):**
$$\text{MI}(x_i, y) = \sum_{x_i \in \{0,1\}} \sum_{y \in \{0,1\}} p(x_i, y) \log \frac{p(x_i, y)}{p(x_i)p(y)} = \text{KL}(p(x_i, y) \| p(x_i)p(y))$$

## 1. Forward Search (Wrapper Method)

**Algorithm:**
1. Initialise $\mathcal{F} = \emptyset$
2. Repeat:
   - For each $i \notin \mathcal{F}$, evaluate $\mathcal{F} \cup \{i\}$ using cross validation
   - Set $\mathcal{F} \leftarrow$ best $\mathcal{F} \cup \{i\}$ found
3. Return the best $\mathcal{F}$ seen across all iterations

Each outer iteration adds one feature. Terminates when $|\mathcal{F}| = n$ or a size threshold is reached.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification

np.random.seed(0)

# Dataset: 5 truly informative features, 15 noise features (n=20 >> m=80)
n_informative = 5
n_total       = 20
X, y = make_classification(
    n_samples=80, n_features=n_total, n_informative=n_informative,
    n_redundant=0, n_repeated=0, random_state=0
)

def forward_search(X, y, max_features=None, cv=5):
    n = X.shape[1]
    selected   = []
    remaining  = list(range(n))
    best_scores = []
    selection_order = []

    if max_features is None:
        max_features = n

    for _ in range(max_features):
        best_score = -np.inf
        best_feat  = None
        for feat in remaining:
            candidate = selected + [feat]
            score = cross_val_score(
                LogisticRegression(max_iter=500), X[:, candidate], y, cv=cv
            ).mean()
            if score > best_score:
                best_score = score
                best_feat  = feat
        selected.append(best_feat)
        remaining.remove(best_feat)
        best_scores.append(best_score)
        selection_order.append(best_feat)

    return selection_order, best_scores

order, scores = forward_search(X, y, max_features=n_total)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: CV accuracy as features are added
ax = axes[0]
ax.plot(range(1, len(scores)+1), scores, 'b-o', lw=2.5, ms=6)
best_k = np.argmax(scores) + 1
ax.axvline(best_k, color='green', ls='--', lw=2,
           label=f'Best: {best_k} features (acc={scores[best_k-1]:.3f})')
ax.set_xlabel('Number of features selected')
ax.set_ylabel('5-fold CV accuracy')
ax.set_title('Forward Search: CV accuracy as features are added')
ax.legend(fontsize=9)

# Right: feature selection order — are informative features picked first?
ax = axes[1]
colors = ['#2166ac' if f < n_informative else '#d6604d' for f in order]
bars = ax.bar(range(1, len(order)+1), [scores[i] for i in range(len(order))],
              color=colors, edgecolor='k', linewidth=0.5)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#2166ac', label='Informative feature'),
    Patch(facecolor='#d6604d', label='Noise feature')
], fontsize=9)
ax.set_xlabel('Selection step'); ax.set_ylabel('CV accuracy after adding feature')
ax.set_title(f'Feature selection order\n(blue = truly informative, red = noise)')
ax.set_xticks(range(1, len(order)+1))
ax.set_xticklabels([f'f{f}' for f in order], rotation=45, fontsize=7)

fig.suptitle('Forward Search (Wrapper Feature Selection)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Backward Search

**Algorithm:** The symmetric reverse of forward search.
1. Initialise $\mathcal{F} = \{1, \ldots, n\}$ (all features)
2. Repeat:
   - For each $i \in \mathcal{F}$, evaluate $\mathcal{F} \setminus \{i\}$ using cross validation
   - Set $\mathcal{F} \leftarrow$ best $\mathcal{F} \setminus \{i\}$ found
3. Return the best $\mathcal{F}$ seen

Forward search is preferred when the true feature set is small. Backward search starts from the full model — useful when removing features is more natural.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification

np.random.seed(0)

n_informative = 5
n_total       = 15
X, y = make_classification(
    n_samples=80, n_features=n_total, n_informative=n_informative,
    n_redundant=0, n_repeated=0, random_state=0
)

def backward_search(X, y, cv=5):
    selected    = list(range(X.shape[1]))
    all_scores  = [cross_val_score(LogisticRegression(max_iter=500),
                                   X[:, selected], y, cv=cv).mean()]
    removal_order = []

    while len(selected) > 1:
        best_score  = -np.inf
        worst_feat  = None
        for feat in selected:
            candidate = [f for f in selected if f != feat]
            score = cross_val_score(
                LogisticRegression(max_iter=500), X[:, candidate], y, cv=cv
            ).mean()
            if score > best_score:
                best_score = score
                worst_feat = feat
        selected.remove(worst_feat)
        removal_order.append(worst_feat)
        all_scores.append(best_score)

    return removal_order, all_scores

removal_order, scores_back = backward_search(X, y)
n_features_remaining = list(range(n_total, 0, -1))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(n_features_remaining, scores_back, 'r-s', lw=2.5, ms=6)
best_k = n_features_remaining[np.argmax(scores_back)]
ax.axvline(best_k, color='green', ls='--', lw=2,
           label=f'Best: {best_k} features (acc={max(scores_back):.3f})')
ax.set_xlabel('Number of features remaining')
ax.set_ylabel('5-fold CV accuracy')
ax.set_title('Backward Search: CV accuracy as features are removed\n'
             f'(started with all {n_total} features, {n_informative} truly informative)')
ax.legend(fontsize=9)
# Annotate first few removed (noise) features
for i, feat in enumerate(removal_order[:4]):
    label = 'noise' if feat >= n_informative else 'info'
    ax.annotate(f'f{feat} ({label})',
                xy=(n_total - i, scores_back[i]),
                xytext=(n_total - i - 1, scores_back[i] - 0.04),
                fontsize=7.5, color='gray',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))
plt.tight_layout()
plt.show()

## 3. Filter Feature Selection: Mutual Information

**Filter methods** score each feature $x_i$ independently using a simple statistic $S(i)$, then select the top $k$ features. No calls to the learning algorithm are needed — $O(n)$ cost.

**Mutual information:**
$$\text{MI}(x_i, y) = \sum_{x_i} \sum_y p(x_i, y) \log \frac{p(x_i, y)}{p(x_i)p(y)}$$

This equals $\text{KL}(p(x_i, y) \| p(x_i)p(y))$ — the KL divergence between the joint distribution and the product of marginals. If $x_i \perp y$, MI = 0. Higher MI → more informative feature.

**Alternatives for continuous features:** Pearson correlation, F-statistic, ANOVA.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

np.random.seed(42)

n_informative = 5
n_total       = 20
X, y = make_classification(
    n_samples=200, n_features=n_total, n_informative=n_informative,
    n_redundant=0, n_repeated=0, random_state=42
)

mi_scores  = mutual_info_classif(X, y, random_state=0)
f_scores, _= f_classif(X, y)
corr_scores= np.abs(np.corrcoef(X.T, y)[-1, :-1])

feat_idx   = np.arange(n_total)
colors     = ['#2166ac' if i < n_informative else '#d6604d' for i in feat_idx]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, scores, title, ylabel in zip(
    axes,
    [mi_scores, f_scores / f_scores.max(), corr_scores],
    ['Mutual Information', 'F-statistic (normalised)', 'Pearson |correlation|'],
    ['MI score', 'Normalised F', '|corr(x_i, y)|']
):
    order = np.argsort(scores)[::-1]
    ax.bar(range(n_total), scores[order],
           color=[colors[i] for i in order], edgecolor='k', linewidth=0.5)
    ax.set_xticks(range(n_total))
    ax.set_xticklabels([f'f{i}' for i in order], rotation=45, fontsize=7)
    ax.set_ylabel(ylabel); ax.set_title(title)
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor='#2166ac', label='Informative'),
        Patch(facecolor='#d6604d', label='Noise')
    ], fontsize=8.5)

fig.suptitle('Filter Feature Scores (blue = truly informative features)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Wrapper vs. Filter: Accuracy-Cost Tradeoff

Wrapper methods call the learning algorithm for every candidate subset — expensive but considers feature interactions.
Filter methods score features in isolation — cheap but may miss interactions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

np.random.seed(0)

n_informative = 5
n_total       = 15
X, y = make_classification(
    n_samples=150, n_features=n_total, n_informative=n_informative,
    n_redundant=2, n_repeated=0, random_state=0
)

# Filter: rank by MI, then try top-k subsets
mi     = mutual_info_classif(X, y, random_state=0)
ranked = np.argsort(mi)[::-1]

filter_scores  = []
k_values       = list(range(1, n_total + 1))
for k in k_values:
    top_k = ranked[:k]
    score = cross_val_score(LogisticRegression(max_iter=500),
                            X[:, top_k], y, cv=5).mean()
    filter_scores.append(score)

# Forward wrapper (reuse function from Section 1)
def forward_search(X, y, max_features=None, cv=5):
    n = X.shape[1]
    selected, remaining = [], list(range(n))
    scores = []
    if max_features is None:
        max_features = n
    for _ in range(max_features):
        best_score, best_feat = -np.inf, None
        for feat in remaining:
            s = cross_val_score(LogisticRegression(max_iter=500),
                                X[:, selected + [feat]], y, cv=cv).mean()
            if s > best_score:
                best_score, best_feat = s, feat
        selected.append(best_feat)
        remaining.remove(best_feat)
        scores.append(best_score)
    return scores

wrapper_scores = forward_search(X, y)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_values, filter_scores,  'r-s', lw=2.5, ms=6, label='Filter (MI ranking)')
ax.plot(k_values, wrapper_scores, 'b-o', lw=2.5, ms=6, label='Wrapper (forward search)')

ax.axvline(k_values[np.argmax(filter_scores)],  color='red',   ls='--', lw=1.5,
           label=f'Filter best k={k_values[np.argmax(filter_scores)]}')
ax.axvline(k_values[np.argmax(wrapper_scores)], color='blue',  ls='--', lw=1.5,
           label=f'Wrapper best k={k_values[np.argmax(wrapper_scores)]}')

ax.set_xlabel('Number of features $k$')
ax.set_ylabel('5-fold CV accuracy')
ax.set_title('Wrapper vs. Filter Feature Selection\n'
             'Wrapper: more accurate, $O(n^2)$ calls. Filter: faster, $O(n)$ calls.')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="464"
     viewBox="0 0 780 464" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>

  <!-- Row 1: High-dimensional input -->
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">n features, n &#x226b; m</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">(high-dim data)</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >VC(H) = O(n) &#x2014; overfitting guaranteed when n &#x226b; m training examples</text>

  <!-- step 1&#x2192;2 -->
  <line x1="102" y1="58" x2="102" y2="108"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="82" font-size="11.5" font-style="italic" fill="#555"
        >choose feature selection strategy</text>

  <!-- Row 2: Score / search features -->
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="135" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Score / search</text>
  <text x="102" y="152" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">features</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >wrapper: CV on subsets (O(n&#xb2;) calls); filter: MI/correlation score (O(n))</text>

  <!-- step 2&#x2192;3 -->
  <line x1="102" y1="158" x2="102" y2="208"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="182" font-size="11.5" font-style="italic" fill="#555"
        >select best subset F &#x2286; {1,...,n}</text>

  <!-- Row 3: Feature subset -->
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="240" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Feature subset F</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >|F| &#x226a; n &#x2014; reduces effective VC dimension from O(n) to O(|F|)</text>

  <!-- step 3&#x2192;4 -->
  <line x1="102" y1="258" x2="102" y2="308"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="282" font-size="11.5" font-style="italic" fill="#555"
        >retrain final model on F using all training data</text>

  <!-- Row 4: Reduced model -->
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="340" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Reduced model</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >lower variance, better generalisation when true signal is sparse</text>

  <!-- step 4&#x2192;5 -->
  <line x1="102" y1="358" x2="102" y2="408"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>

  <!-- Row 5: Prediction -->
  <rect x="10" y="412" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="440" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Deploy model</text>
  <line x1="197" y1="435" x2="216" y2="435"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="412" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="440" font-size="13" text-anchor="middle" fill="#333"
        >predict using x_F only &#x2014; robust to irrelevant features at inference time</text>
</svg>
""")

## Summary

| Method | Strategy | Cost | Captures interactions? |
|---|---|---|---|
| Forward search (wrapper) | Start empty; add best feature each step | $O(n^2)$ learner calls | Yes |
| Backward search (wrapper) | Start full; remove worst feature each step | $O(n^2)$ learner calls | Yes |
| Filter (MI/correlation) | Score each feature independently; pick top $k$ | $O(n)$ | No |

| Concept | Formula / Insight | Key point |
|---|---|---|
| Why select features | $\text{VC}(\mathcal{H}) = O(n)$; if $n \gg m$ → overfit | Reduce effective complexity |
| Mutual information | $\text{MI}(x_i, y) = \text{KL}(p(x_i,y) \| p(x_i)p(y))$ | Zero iff $x_i \perp y$; higher → more informative |
| Wrapper accuracy | Uses the actual learner for evaluation | Better quality; expensive |
| Filter speed | No learner calls; $O(n)$ passes | Cheap; may miss synergies |

**Key insight:** when $n \gg m$, even a simple linear classifier overfits because its VC dimension is $O(n)$; feature selection reduces effective dimension from $O(n)$ to $O(|\mathcal{F}|)$, bringing sample complexity back under control.